# Chunking Strategis

## Fixed-Size Chunking
Simply decide number of tokens in our chunk

In [1]:
from pathlib import Path

# Simple fixed-size token chunking with overlap
# Here, "token" means a whitespace-separated word.
def load_markdown_docs(folder='sample-documents'):
    docs = []
    for path in Path(folder).rglob('*.md'):
        text = path.read_text(encoding='utf-8', errors='ignore').strip()
        docs.append({'id': path.stem, 'path': str(path), 'text': text})
    return docs


def chunk_by_tokens(text, chunk_tokens=120, overlap_tokens=20):
    tokens = text.split() # Split text into tokens (words)
    if len(tokens) <= chunk_tokens: # if text is shorter than chunk size, return as is
        return [' '.join(tokens)]

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_tokens # Calculate end index for the chunk
        chunks.append(' '.join(tokens[start:end])) # Join tokens back into text chunk
        if end >= len(tokens):
            break
        start = max(0, end - overlap_tokens)
    return chunks


docs = load_markdown_docs('sample-documents')
for doc in docs:
    chunks = chunk_by_tokens(doc['text'], chunk_tokens=120, overlap_tokens=20) #set chunk size and overlap here
    print(f"\nFILE: {doc['path']}") # Print file path
    print(f"chunks: {len(chunks)}") # Print number of chunks created
    print(chunks[0][:1000]) # Print first 1000 characters of the first chunk for preview

# you can see the chunk cut points are not very smart, they just split by token count regardless of sentence boundaries or meaning.


FILE: sample-documents\hometown.md
chunks: 8
--- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags: [hometown, local-history, guide] --- # My Hometown Overview -------- My hometown is a medium-sized city located in the temperate region of the country. It combines a long history with modern amenities and a friendly community. The town is known for its tree-lined streets, a historic downtown area, and a mix of industry, small businesses, and agriculture in the surrounding countryside. Quick facts - Population: ~75,000 (approx.) - Region: Central valley - Founded: 1800s (historic settlement) - Language(s): Primary local language and common secondary languages History ------- The town began as a small trading post in the 19th century and grew with the arrival of the railroad. Early industries

FILE: sample-documents\university.md
chunks: 7
--- title: "My University — A Story of Studying Sustainability" author: "John Doe" source: "personal" create

## Content-aware Chunking
adhere to the structure to help inform the meaning of our chunks

### Simple Sentence and Paragraph Splitting

In [2]:
import re
from pathlib import Path
from typing import List

In [3]:
# Simple sentence & paragraph splitting for Markdown files

# ----------------------
# Paragraph splitting
# ----------------------
def paragraph_split(markdown_text: str) -> List[str]:
    """
    Split markdown into paragraphs by blank lines.
    Keeps paragraphs simple and human-readable.
    """
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n+', markdown_text) if p.strip()]
    return paragraphs

In [4]:
# ----------------------
# Naive sentence split
# ----------------------
_sentence_re = re.compile(r'(?<=[.!?])\s+')

def naive_sentence_split(text: str) -> List[str]:
    """
    Very simple sentence splitter:
    - Splits on punctuation (., !, ?) followed by whitespace.
    - Good for quick demos but will make mistakes on abbreviations.
    """
    sents = [s.strip() for s in _sentence_re.split(text) if s.strip()]
    if not sents:
        # fallback: split on newlines
        sents = [line.strip() for line in text.splitlines() if line.strip()]
    return sents

In [5]:
# ----------------------
# NLTK-based sentence splitting (recommended for many cases)
# ----------------------


def nltk_sentence_split(text: str) -> List[str]:
    """
    Uses NLTK's Punkt sentence tokenizer.
    Install with: pip install nltk
    First run: import nltk; nltk.download('punkt')
    """
    import nltk
    from nltk.tokenize import sent_tokenize

    # [first run only -- uncomment this]
    nltk.download('punkt')

    return sent_tokenize(text)

In [6]:
# ----------------------
# spaCy-based sentence splitting (more robust, needs model)
# ----------------------
def spacy_sentence_split(text: str, model: str = "en_core_web_sm") -> List[str]:
    """
    Uses spaCy for sentence segmentation.
    Install with: pip install spacy
    Then download model: python -m spacy download en_core_web_sm
    spaCy often gives more linguistically-aware splits.
    """
    
    import spacy
    nlp = spacy.load(model)
    
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

In [7]:
# ----------------------
# Demo: run on all .md files in `rag/sample-documents`
# ----------------------

import textwrap
from pathlib import Path

def _print_header(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def _print_sub(title: str):
    print("\n" + "-" * 60)
    print(title)
    print("-" * 60)

def _short(s: str, width: int = 160):
    return textwrap.shorten(s.replace("\n", " "), width=width, placeholder=" ...")

def demo_splitters_pretty(folder: str = "sample-documents",
                          max_preview_paras: int = 2,
                          max_preview_sents: int = 3,
                          wrap_width: int = 100):
    """
    Pretty demo that lists paragraphs and sentence-splits (naive / NLTK / spaCy).
    Designed to be readable in notebook outputs.
    """
    p = Path(folder)
    md_files = sorted(p.glob("*.md"))
    if not md_files:
        print(f"No markdown files found in {folder!r}")
        return

    for f in md_files:
        text = f.read_text(encoding="utf8")
        _print_header(f"FILE: {f.name}")

        paras = paragraph_split(text)
        print(f"Paragraphs: {len(paras)} (showing first {min(max_preview_paras, len(paras))})")
        for i, para in enumerate(paras[:max_preview_paras], start=1):
            print(f"\nParagraph {i}:")
            print(textwrap.fill(_short(para, width=400), width=wrap_width))

        # pick a paragraph for sentence demos (prefer first non-empty)
        first_para = next((p for p in paras if p.strip()), text)

        _print_sub("Naive sentence split")
        naive_sents = naive_sentence_split(first_para)
        for i, s in enumerate(naive_sents[:max_preview_sents], start=1):
            print(f" {i}. {textwrap.fill(_short(s, width=200), width=wrap_width)}")
        if len(naive_sents) > max_preview_sents:
            print(f"  ... (+{len(naive_sents)-max_preview_sents} more)")

        _print_sub("NLTK")
        try:
            nltk_sents = nltk_sentence_split(first_para)
            for i, s in enumerate(nltk_sents[:max_preview_sents], start=1):
                print(f" {i}. {textwrap.fill(_short(s, width=200), width=wrap_width)}")
            if len(nltk_sents) > max_preview_sents:
                print(f"  ... (+{len(nltk_sents)-max_preview_sents} more)")
        except Exception:
            print(" NLTK not available. Install: pip install nltk; python -c \"import nltk; nltk.download('punkt')\"")

        _print_sub("spaCy")
        try:
            spacy_sents = spacy_sentence_split(first_para)
            for i, s in enumerate(spacy_sents[:max_preview_sents], start=1):
                print(f" {i}. {textwrap.fill(_short(s, width=200), width=wrap_width)}")
            if len(spacy_sents) > max_preview_sents:
                print(f"  ... (+{len(spacy_sents)-max_preview_sents} more)")
        except Exception:
            print(" spaCy not available. Install: pip install spacy; python -m spacy download en_core_web_sm")


demo_splitters_pretty()


FILE: hometown.md
Paragraphs: 34 (showing first 2)

Paragraph 1:
--- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

Paragraph 2:
# My Hometown

------------------------------------------------------------
Naive sentence split
------------------------------------------------------------
 1. --- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

------------------------------------------------------------
NLTK
------------------------------------------------------------


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\radyadhewa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


 1. --- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

------------------------------------------------------------
spaCy
------------------------------------------------------------


c:\Users\radyadhewa\Storage\code\personal\AI-Engineer-RoadmapSH-Code\.venv\Lib\site-packages\spacy\util.py:969: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


 1. --- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

FILE: university.md
Paragraphs: 29 (showing first 2)

Paragraph 1:
--- title: "My University — A Story of Studying Sustainability" author: "John Doe" source:
"personal" created: "2026-03-21" tags: [university, sustainability, education, story] ---

Paragraph 2:
# My University — A Sustainability Story

------------------------------------------------------------
Naive sentence split
------------------------------------------------------------
 1. --- title: "My University — A Story of Studying Sustainability" author: "John Doe" source:
"personal" created: "2026-03-21" tags: [university, sustainability, education, story] ---

------------------------------------------------------------
NLTK
------------------------------------------------------------
 1. --- title: "My University — A Story of Studying Sustainability" author: "John Doe" source:
"pers

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\radyadhewa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


 1. --- title: "My University — A Story of Studying Sustainability" author: "John Doe" source:
"personal" created: "2026-03-21" tags:
 2. [university, sustainability, education, story] ---


### LangChain's RecursiveCharacterTextSplitter  

RecursiveCharacterTextSplitter is a LangChain text-splitting utility that breaks long text into chunks by attempting to split on progressively smaller separators (paragraphs → lines → spaces → characters) until each chunk fits the target size.

[More Explanation](https://dev.to/eteimz/understanding-langchains-recursivecharactertextsplitter-2846)

#### How it Works

- Top‑down separators: It tries the largest separators first (e.g., "\n\n", then "\n", then " ", then "") to keep semantic boundaries intact.
- Recursion: If a block is still too large after splitting by a separator, it recurses to the next, smaller separator.
- Overlap: Produces overlapping chunks controlled by chunk_overlap so nearby context is preserved across chunks.
- Deterministic: Behavior is deterministic and fast, requiring no ML models.

#### Key Params


- chunk_size: target max characters per chunk.
- chunk_overlap: number of characters to overlap between consecutive chunks.
- separators: ordered list of separators tried from largest → smallest.

#### Codebase

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\radyadhewa\Storage\code\personal\AI-Engineer-RoadmapSH-Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
docs = load_markdown_docs('sample-documents')
for doc in docs:
    text = doc['text']
    print(f"\nFILE: {doc['path']}")
    print(f"Original length: {len(text)} characters")

    # Create a RecursiveCharacterTextSplitter instance
    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
    
    # Split the text into chunks
    chunks = splitter.create_documents([text])
    
    print(f"Number of chunks created: {len(chunks)}")
    print(f"chunk preview: {chunks[5].page_content[:1000]}")

    # it not merely split by token count, but tries to split at natural boundaries (sentences, paragraphs) while respecting the chunk size and overlap.


FILE: sample-documents\hometown.md
Original length: 5516 characters
Number of chunks created: 45
chunk preview: and later small-scale manufacturing. The historic district preserves several 19th- and early-20th-century buildings, including the old courthouse, the market hall, and several churches.

FILE: sample-documents\university.md
Original length: 5292 characters
Number of chunks created: 49
chunk preview: Why Sustainability?
-------------------


### Document structure-based chunking

pdf -> can be based on headers / text / tables  
html -> based on tags  
markdown -> recognizing markdown syntax (headings, lists, code blocks)  
LaTex -> LaTex commands and environment

In [27]:
# Heading-aware chunking for Markdown files
# Strategy:
# 1) Split document into sections by Markdown headings (keep heading with its content)
# 2) If a section is too long, split it into paragraph-sized chunks
# 3) Optionally add a small overlap so nearby context isn't lost

def paragraph_split(text: str) -> List[str]:
    """Helper to split text into paragraphs."""
    # Split by double newlines, ignoring empty strings
    return [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

def chunk_by_headings(text: str, max_chars: int = 800) -> List[str]:
    """
    Chunks markdown by heading. 
    If a section is too long, splits by paragraph.
    Injects the heading into sub-chunks so context isn't lost.
    """
    # 1. Separate Frontmatter from the main body
    frontmatter = ""
    body = text
    if text.startswith("---"):
        parts = text.split("---", 2)
        if len(parts) >= 3:
            frontmatter = f"---{parts[1]}---"
            body = parts[2].strip()

    # 2. Split body by headings
    # (?m) = multiline, ^ = start of line, #{1,6} = 1 to 6 hashes
    parts = re.split(r'(?m)^(#{1,6}\s+.*)$', body)
    
    sections = []
    
    # Handle text before the first heading (if any)
    if parts[0].strip():
        sections.append(("", parts[0].strip()))

    # Group headings with their content
    for i in range(1, len(parts), 2):
        heading = parts[i].strip()
        content = parts[i+1].strip() if i+1 < len(parts) else ''
        sections.append((heading, content))

    chunks = []
    
    # Optional: Add frontmatter as its own chunk if it exists
    if frontmatter:
        chunks.append(frontmatter)

    # 3. Process each section
    for heading, content in sections:
        full_section = f"{heading}\n\n{content}".strip()
        
        # If the whole section fits, add it
        if len(full_section) <= max_chars:
            if full_section:
                chunks.append(full_section)
            continue
            
        # 4. If it's too long, split into paragraphs
        paras = paragraph_split(content)
        current_chunk = heading # Start the chunk with the heading
        
        for p in paras:
            # Check if adding the next paragraph exceeds the limit
            if len(current_chunk) + len(p) + 2 <= max_chars:
                # First paragraph after heading just needs \n\n
                current_chunk += f"\n\n{p}" 
            else:
                # Chunk is full. Save it.
                if current_chunk != heading: # Don't save empty headings
                    chunks.append(current_chunk.strip())
                # Start a new chunk, repeating the heading for context
                current_chunk = f"{heading} (Continued)\n\n{p}"
                
        # Catch any leftover text
        if current_chunk and current_chunk != heading:
            chunks.append(current_chunk.strip())

    return chunks

In [13]:
# Demo: run on files in sample-documents (keeps output short for notebooks)
p = Path('sample-documents')
md = sorted(p.glob('*.md'))

for f in md:
    txt = f.read_text(encoding='utf8')
    chunks = chunk_by_headings(txt, max_chars=800)
    print('\n' + '='*50)
    print(f'FILE: {f.name} — chunks: {len(chunks)}')
    for i, c in enumerate(chunks[:3], start=1):
        print('\n-- chunk %d --' % i)
        print(c[:300].replace('\n', ' ') + ('...' if len(c) > 300 else ''))


FILE: hometown.md — chunks: 9

-- chunk 1 --
--- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags: [hometown, local-history, guide] ---

-- chunk 2 --
# My Hometown  Overview --------  My hometown is a medium-sized city located in the temperate region of the country. It combines a long history with modern amenities and a friendly community. The town is known for its tree-lined streets, a historic downtown area, and a mix of industry, small busines...

-- chunk 3 --
# My Hometown (Continued)  The town began as a small trading post in the 19th century and grew with the arrival of the railroad. Early industries included milling, agriculture, and later small-scale manufacturing. The historic district preserves several 19th- and early-20th-century buildings, includ...

FILE: university.md — chunks: 9

-- chunk 1 --
--- title: "My University — A Story of Studying Sustainability" author: "John Doe" source: "personal" created: "2026-03-21" tags: [un

### Semantic Chunking



Semantic Chunking is a smart way of breaking down a document based on its meaning (semantics) rather than arbitrary rules like character counts or punctuation.

**How it Works (in 4 Simple Steps)**

1. Break it Down: First, we split the whole text into individual sentences.

2. Translate to Math: We use an Embedding Model (like your Hugging Face model) to turn every single sentence into a vector (a list of numbers representing its "vibe").

3. Compare Neighbors: The computer compares Sentence 1 to Sentence 2.
    - Are they talking about the same thing? (High similarity).
    - Keep them together.

4. Make the Cut: The computer compares Sentence 2 to Sentence 3.
    - Did the topic suddenly shift from "Apples" to "Cars"? (Low similarity / The "Threshold").
    - Make a cut right there and start a new chunk!

In [28]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import InferenceClient
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [16]:
client = InferenceClient(token=hf_token)
MODEL_ID = "sentence-transformers/all-mpnet-base-v2"

In [17]:
def get_hf_embedding(text: str) -> List[float]:
    """Calls Hugging Face API to get a real 768-dimensional embedding."""
    # The API expects a list of strings, we send a list with one item
    response = client.feature_extraction([text], model=MODEL_ID)
    # The response is a numpy array-like object. We return the first (and only) vector.
    return np.array(response)[0].tolist()

def naive_sentence_split(text: str) -> List[str]:
    """Helper function to split text into sentences."""
    # Split on ., !, or ? followed by a space or newline
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

def semantic_chunk_simple(text: str, 
                          embedding_func,
                          threshold: float = 0.5) -> List[str]:
    """
    Splits text by semantic meaning rather than just length.
    """
    sentences = naive_sentence_split(text)
    
    if len(sentences) <= 1:
        return sentences
    
    print(f"Generating embeddings for {len(sentences)} sentences... please wait.")
    
    # Generate embeddings for each sentence
    embeddings = [embedding_func(sent) for sent in sentences]
    embeddings = np.array(embeddings)
    
    chunks = []
    current_chunk = [sentences[0]]
    
    # Compare consecutive sentences
    for i in range(1, len(sentences)):
        # Calculate cosine similarity (-1 to 1). 1 means identical meaning.
        # reshape(1, -1) is required by sklearn for single samples
        sim = cosine_similarity(embeddings[i-1].reshape(1, -1), 
                                embeddings[i].reshape(1, -1))[0][0]
        
        # If similarity is lower than threshold, the topic changed. Start a new chunk.
        if sim < threshold:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
        else:
            # Topic is the same, keep adding to current chunk
            current_chunk.append(sentences[i])
            
    # Add the last chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))
        
    return chunks

In [21]:
# ============================================================================
# Demo on files in sample-documents
# ============================================================================
print("Starting Semantic Chunking Demo...")

p = Path('sample-documents')
md_files = sorted(p.glob('*.md'))

if not md_files:
    print('No markdown files found in sample-documents folder.')
else:
    # We will just process the first file
    f = md_files[0]
    txt = f.read_text(encoding='utf8')

    print(f"\nProcessing File: {f.name}")
    
    # Note: A threshold of 0.5 might need tuning depending on the text. 
    # MPNet vectors often have high similarity scores generally.
    semantic_chunks = semantic_chunk_simple(
        txt,
        embedding_func=get_hf_embedding,
        threshold=0.6 # Adjust this up or down to change chunk sizes
    )

    print(f"\nSemantic chunks created: {len(semantic_chunks)}")
    print("="*50)
    for i, chunk in enumerate(semantic_chunks, start=1):
        print(f"\n--- Chunk {i} ---")
        print(chunk)

Starting Semantic Chunking Demo...

Processing File: hometown.md
Generating embeddings for 38 sentences... please wait.

Semantic chunks created: 37

--- Chunk 1 ---
---
title: "Hometown Overview"
author: "Your Name"
source: "personal"
created: "2026-03-21"
tags: [hometown, local-history, guide]
---

# My Hometown

Overview
--------

My hometown is a medium-sized city located in the temperate region of the country.

--- Chunk 2 ---
It combines a long history with modern amenities and a friendly community.

--- Chunk 3 ---
The town is known for its tree-lined streets, a historic downtown area, and a mix of industry, small businesses, and agriculture in the surrounding countryside.

--- Chunk 4 ---
Quick facts
- Population: ~75,000 (approx.)
- Region: Central valley 
- Founded: 1800s (historic settlement)
- Language(s): Primary local language and common secondary languages

History
-------

The town began as a small trading post in the 19th century and grew with the arrival of the railro

### Contextual Chunking with LLMs

In [29]:
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [23]:
def get_chunk_context(whole_document: str, chunk: str) -> str:
    """
    Uses an LLM to generate a situational context for a specific chunk 
    based on the entire document.
    """
    # This is the exact prompt structure recommended by Anthropic
    prompt = f"""
    <document>
    {whole_document}
    </document>

    Here is the chunk we want to situate within the whole document:
    <chunk>
    {chunk}
    </chunk>

    Please give a short, succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk. 
    Answer ONLY with the succinct context and nothing else.
    """

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": "z-ai/glm-4.5-air:free", # Using your preferred free model
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0 # We want facts, not creativity
    }

    try:
        response = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers, json=payload)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content'].strip()
    except Exception as e:
        print(f"Error generating context: {e}")
        return ""

def contextualize_chunks(whole_document: str, raw_chunks: List[str]) -> List[str]:
    """
    Takes raw chunks, gets context for each, and prepends it.
    """
    contextualized_chunks = []
    
    print(f"Contextualizing {len(raw_chunks)} chunks... This might take a moment.")
    
    for i, chunk in enumerate(raw_chunks, 1):
        print(f"Processing chunk {i}/{len(raw_chunks)}...")
        
        # 1. Ask the LLM for the context
        context = get_chunk_context(whole_document, chunk)
        
        # 2. Prepend the context to the original chunk
        # Format: "Context: [LLM Output]\n\n[Original Chunk]"
        augmented_chunk = f"Context: {context}\n\n{chunk}"
        contextualized_chunks.append(augmented_chunk)
        
    return contextualized_chunks

In [26]:
# ============================================================================
# Demo Usage with Real Documents
# ============================================================================
def simple_paragraph_chunker(text: str, max_chunks: int = 10) -> list[str]:
    """A basic paragraph chunker just for the demo."""
    # Split by double newlines to get paragraphs
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    # Return only a few chunks to prevent long API wait times during testing
    return paragraphs[:max_chunks]

print("Starting Contextual Chunking Demo...")

p = Path('sample-documents')
md_files = sorted(p.glob('*.md'))

# 1. Pick the first file
target_file = md_files[0]
print(f"\nReading {target_file.name}...")
full_document_text = target_file.read_text(encoding='utf8')

# 2. Get raw chunks 
# (you can replace this with your Heading or Semantic chunker)
raw_chunks = simple_paragraph_chunker(full_document_text, max_chunks=5) # example for 5 chunks

print(f"Extracted {len(raw_chunks)} chunks for testing.")

# 3. Run the contextualizer
final_chunks = contextualize_chunks(full_document_text, raw_chunks)

print("\n" + "="*50)
for i, chunk in enumerate(final_chunks, 1):
    print(f"\n--- Final Contextualized Chunk {i} ---")
    print(chunk)

Starting Contextual Chunking Demo...

Reading hometown.md...
Extracted 5 chunks for testing.
Contextualizing 5 chunks... This might take a moment.
Processing chunk 1/5...
Processing chunk 2/5...
Processing chunk 3/5...
Processing chunk 4/5...
Processing chunk 5/5...


--- Final Contextualized Chunk 1 ---
Context: This YAML frontmatter chunk appears at the beginning of the document and contains metadata for the entire hometown overview, including title, author, source, creation date, and tags used for categorization.

---
title: "Hometown Overview"
author: "Your Name"
source: "personal"
created: "2026-03-21"
tags: [hometown, local-history, guide]
---

--- Final Contextualized Chunk 2 ---
Context: This is the main title heading for a comprehensive document about a medium-sized hometown that combines historical significance with modern amenities. The document provides an overview of the town's characteristics, history, geography, culture, economy, transportation, education, attractions, l